# Association of cluster with outcome

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from utils.utils import load_encrypted_xlsx
import os
os.environ["R_HOME"] = "/Library/Frameworks/R.framework/Versions/4.1/Resources"
from pymer4.models import Lmer
import statsmodels.api as sm

In [ ]:
registry_path = '/Users/jk1/Library/CloudStorage/OneDrive-unige.ch/icu_research/dci_sah/data/sos_sah_data/post_hoc_modified_aSAH_DATA_2009_2023_24122023.xlsx'
outcome_data_path = '/Users/jk1/Library/CloudStorage/OneDrive-unige.ch/icu_research/dci_sah/data/sos_sah_data/follow_up/aSAH_DATA_2009_2024_18122024.xlsx'
bp_path = "/Users/jk1/Library/CloudStorage/OneDrive-unige.ch/icu_research/dci_sah/data/pdms_data/extracted_data/20240116_SAH_SOS_Blutdruecke.csv"
registry_pdms_correspondence_path = "/Users/jk1/Library/CloudStorage/OneDrive-unige.ch/icu_research/dci_sah/data/pdms_data/registry_pdms_correspondence.csv"
cluster_path = '/Users/jk1/Downloads/daily_order4_clip21d_gbtm_4cluster_map.csv'

In [ ]:
registry_df = load_encrypted_xlsx(registry_path)
bp_df = pd.read_csv(bp_path, sep=';', decimal='.')
registry_pdms_correspondence_df = pd.read_csv(registry_pdms_correspondence_path)
outcome_df = load_encrypted_xlsx(outcome_data_path)
cluster_df = pd.read_csv(cluster_path)

In [ ]:
bp_df = bp_df.merge(registry_pdms_correspondence_df, how="left", on="pNr")

In [ ]:
# <!-- fillna in mRS_FU_1y with mRS_2FU_2y and then mRS_3FU_5y -->
outcome_df['mRS_FU_1y'] = outcome_df['mRS_FU_1y'].fillna(outcome_df['mRS_2FU_2y'])
outcome_df['mRS_FU_1y'] = outcome_df['mRS_FU_1y'].fillna(outcome_df['mRS_3FU_5y'])

outcome_df['mRS_FU_1y'] = pd.to_numeric(outcome_df['mRS_FU_1y'], errors='coerce')
outcome_df['mRS_discharge'] = pd.to_numeric(outcome_df['mRS_discharge'], errors='coerce')

# if mRS_discharge == 6, set mRS_FU_1y to 6
outcome_df.loc[outcome_df['mRS_discharge'] == 6, 'mRS_FU_1y'] = 6

In [ ]:
outcome_df["mRS_FU_1y_int"] = pd.to_numeric(outcome_df["mRS_FU_1y"], errors="coerce")
bp_df["Date_birth"] = pd.to_datetime(bp_df["Date_birth"], format="%d.%m.%Y")
outcome_df["Date_birth"] = pd.to_datetime(outcome_df["Date_birth"])
registry_df['Fisher_Score'] = pd.to_numeric(registry_df['Fisher_Score'], errors='coerce')

In [ ]:
bp_df["mrs_1y"] = np.nan
for pnr in tqdm(bp_df["pNr"].unique()):
    sos_center_nr = bp_df[bp_df["pNr"] == pnr]["SOS-CENTER-YEAR-NO."].values[0]
    name = bp_df[bp_df["pNr"] == pnr]["JoinedName"].values[0]
    date_birth = bp_df[bp_df["pNr"] == pnr]["Date_birth"].values[0]
    mrs_values = outcome_df[(outcome_df["SOS-CENTER-YEAR-NO."] == sos_center_nr) &
                        (outcome_df["Name"] == name) &
                        (outcome_df["Date_birth"] == date_birth)]["mRS_FU_1y_int"]
    if len(mrs_values) == 0:
        mrs = np.nan
    else:
        mrs = mrs_values.values[0]

    bp_df.loc[bp_df["pNr"] == pnr, "mrs_1y"] = mrs

In [ ]:
registry_df = registry_df.drop_duplicates(subset=["SOS-CENTER-YEAR-NO.", "Date_birth", "Name"])
bp_df = bp_df.merge(registry_df[["SOS-CENTER-YEAR-NO.", "Date_birth", "Name", 'DCI_ischemia', 'Fisher_Score', 'WFNS'	]], how="left", left_on=["SOS-CENTER-YEAR-NO.", "Date_birth", "JoinedName"],
                right_on=["SOS-CENTER-YEAR-NO.", "Date_birth", "Name"])
# dichotomize mrs_1y into 0-2 and 3-6
bp_df["mrs_1y_02"] = bp_df["mrs_1y"].isin([0, 1, 2]).astype(int)

In [ ]:
# get first measure (by timeBd) for every pNr
first_measure_df = bp_df.groupby("pNr").agg({"timeBd": "min"}).reset_index()
first_measure_df = first_measure_df.rename(columns={"timeBd": "first_timeBd"})
bp_df = bp_df.merge(first_measure_df, how="left", on="pNr")
bp_df["relative_timeBd"] = (pd.to_datetime(bp_df["timeBd"]) - pd.to_datetime(bp_df["first_timeBd"])).dt.total_seconds() / 3600
bp_df["relative_timeBd_days"] = bp_df["relative_timeBd"] / 24
bp_df["relative_timeBd_days_cat"] = bp_df["relative_timeBd_days"].apply(np.floor)
bp_df["relative_timeBd_hours_cat"] = bp_df["relative_timeBd"].apply(np.floor)

In [ ]:
cluster_df = cluster_df.merge(bp_df[["pNr", "mrs_1y", "mrs_1y_02", "DCI_ischemia", "Fisher_Score"]], how="left", on="pNr").drop_duplicates()

In [ ]:
hourly_bp_df = bp_df.groupby(["pNr", "relative_timeBd_hours_cat"]).agg({"systole": "median", "diastole": "median", "mitteldruck": "median", "mrs_1y": "first", "mrs_1y_02": "first", "DCI_ischemia": "first"}).reset_index()
hourly_bp_df = hourly_bp_df.rename(columns={"relative_timeBd_hours_cat": "relative_timeBd_hours"})
hourly_bp_df = hourly_bp_df.drop_duplicates(subset=["pNr", "relative_timeBd_hours"])

# get median daily bp for each patient
daily_bp_df = bp_df.groupby(["pNr", "relative_timeBd_days_cat"]).agg({"systole": "median", "diastole": "median", "mitteldruck": "median", "mrs_1y": "first", "mrs_1y_02": "first", "DCI_ischemia": "first"}).reset_index()
daily_bp_df = daily_bp_df.rename(columns={"relative_timeBd_days_cat": "relative_timeBd_days"})
daily_bp_df = daily_bp_df.drop_duplicates(subset=["pNr", "relative_timeBd_days"])

daily_bp_df = daily_bp_df.merge(cluster_df[["pNr", "Cluster"]], how="left", on="pNr")
hourly_bp_df = hourly_bp_df.merge(cluster_df[["pNr", "Cluster"]], how="left", on="pNr")

In [ ]:
cluster_df.Cluster.value_counts()
# plot cluster distribution
plt.figure(figsize=(10, 6))
order = cluster_df.Cluster.unique().sort()
sns.countplot(data=cluster_df, x="Cluster", order=order, palette="Set1", hue="Cluster")
plt.title("Cluster distribution")
plt.xlabel("Cluster")
plt.ylabel("Count")
plt.xticks(rotation=45)

In [ ]:

# plot bp evoloution in each cluster
fig, ax = plt.subplots(figsize=(15, 6))
# ax = sns.kdeplot(data=daily_bp_df, x="relative_timeBd_days", y="systole", hue="Cluster", fill=True, common_norm=False, palette="crest", alpha=0.5)
sns.lineplot(data=daily_bp_df, x="relative_timeBd_days", y="systole", hue="Cluster", palette="Set1", ax=ax)
ax.set_xlim(-1, 22)
# rotate x labels
plt.xticks(rotation=45)

In [ ]:
# plot cluster vs mrs_1y
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=cluster_df, x="Cluster", y="mrs_1y", hue="Cluster", palette="Set3", ax=ax)
ax.set_xlabel("Cluster")
ax.set_ylabel("mRS 1y")

In [ ]:
# ordinal logistic regression
from pyexpat import model
from statsmodels.miscmodels.ordinal_model import OrderedModel
from statsmodels.tools import add_constant
from statsmodels.formula.api import ols


# create a new dataframe with only the relevant columns
# temp_df = cluster_df[["Cluster", "mrs_1y", "Fisher_Score"]].copy()
temp_df = cluster_df[["Cluster", "mrs_1y"]].copy()
temp_df = temp_df.dropna()
temp_df["Cluster"] = temp_df["Cluster"].astype("category")

# model = OrderedModel(temp_df["mrs_1y"], temp_df[["Cluster", "Fisher_Score"]], distr="logit")
model = OrderedModel(temp_df["mrs_1y"], temp_df[["Cluster"]], distr="logit")
results = model.fit(method="bfgs", maxiter=1000, disp=True)
print(results.summary())